**News Topic Classifier Using BERT**

This notebook demonstrates how to fine-tune bert-base-uncased on the AG News dataset to classify news headlines into four categories: World, Sports, Business, and Sci/Tech. It also includes a live deployment block using Gradio.

In [1]:
# Install required libraries
!pip install -q transformers datasets evaluate accelerate gradio scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00


# 1. Load and Explore the Dataset
We use the Hugging Face datasets library to fetch the AG News dataset.

In [2]:
import pandas as pd
from datasets import load_dataset

# Load AG News dataset
dataset = load_dataset("ag_news")

# Inspect dataset structure
print(dataset)

# Display class names
labels = dataset["train"].features["label"].names
print(f"\nTopic Categories: {labels}")

# View a sample row
print("\nSample Data Point:")
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Topic Categories: ['World', 'Sports', 'Business', 'Sci/Tech']

Sample Data Point:
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


# 2. **Tokenization and Preprocessing**
  We initialize the BERT tokenizer to convert text headlines into token IDs and attention masks that the transformer expects.

In [3]:
from transformers import AutoTokenizer

# Load the BERT tokenizer
model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

# Tokenization function
def tokenize_function(examples):
    # AG News has a 'text' column containing the headlines/articles
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Tokenize the dataset in batches
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Rename column to match Trainer expectations
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Select a subset for faster training during your internship testing (Optional)
# Remove .select() if you want to train on the entire dataset
train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(10000))
test_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(2000))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

# 3. Define Evaluation Metrics
We setup a function to calculate Accuracy and F1-score during training using the evaluate library.

In [4]:
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]

    return {"accuracy": acc, "f1": f1}

# 4. Fine-Tune the BERT Model
We set up the configuration, load the pre-trained Sequence Classification model, and run the Hugging Face Trainer.

In [8]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Load BERT for sequence classification with 4 output labels
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=4)

# Define training hyperparameters
training_args = TrainingArguments(
    output_dir="./bert-news-classifier",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch", # Changed from evaluation_strategy to eval_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="none" # Prevents wandb prompts
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Start training
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.280823,0.260290,0.911000,0.910874
2,0.173720,0.268937,0.919500,0.919547


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1250, training_loss=0.2832632034301758, metrics={'train_runtime': 509.9376, 'train_samples_per_second': 39.22, 'train_steps_per_second': 2.451, 'total_flos': 1315578900480000.0, 'train_loss': 0.2832632034301758, 'epoch': 2.0})

# 5. Evaluate the Final Model
We evaluate the fine-tuned model against the test set to confirm performance.

In [9]:
# Evaluate the model
eval_results = trainer.evaluate()
print(f"\nEvaluation Results: {eval_results}")

# Save the fine-tuned model and tokenizer locally
model.save_pretrained("./saved_model")
tokenizer.save_pretrained("./saved_model")


Evaluation Results: {'eval_loss': 0.26012635231018066, 'eval_accuracy': 0.911, 'eval_f1': 0.9108736914804427, 'eval_runtime': 14.8496, 'eval_samples_per_second': 134.684, 'eval_steps_per_second': 8.418, 'epoch': 2.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_model/tokenizer_config.json', './saved_model/tokenizer.json')

# 6. Deployment via Gradio
This script creates an interactive UI directly inside the notebook where you can type or paste news headlines to get instant topic predictions.

In [10]:
import gradio as gr
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Load the saved model and tokenizer
saved_model_path = "./saved_model"
deployed_model = AutoModelForSequenceClassification.from_pretrained(saved_model_path)
deployed_tokenizer = AutoTokenizer.from_pretrained(saved_model_path)

# Label mapping based on AG News documentation
id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

def predict_news_topic(text):
    # Tokenize input text
    inputs = deployed_tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding=True)

    # Get predictions
    with torch.no_grad():
        outputs = deployed_model(**inputs)
        logits = outputs.logits
        probabilities = torch.nn.functional.softmax(logits, dim=-1)[0]

    # Format results for Gradio's Label component
    return {id2label[i]: float(probabilities[i]) for i in range(4)}

# Build the Gradio Interface
demo = gr.Interface(
    fn=predict_news_topic,
    inputs=gr.Textbox(lines=3, placeholder="Enter a news headline here...", label="News Headline"),
    outputs=gr.Label(num_top_classes=4, label="Topic Probabilities"),
    title="News Topic Classifier",
    description="Fine-tuned BERT model classifying headlines into World, Sports, Business, or Sci/Tech.",
    examples=[
        ["Tech giants look to secure energy grid partnerships for new massive AI datacenters."],
        ["The underdog team clinches the championship with a stunning last-minute goal."],
        ["Central banks hint at shifting interest rates amid changing global market trends."]
    ]
)

# Launch the app inline inside your notebook
demo.launch(share=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://af6ac072eddea8d791.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
